# Intro

This notebook gives an intro to our project and its components. It also showcases what can be done

## Load & Inspect OCELs

First, we can load ocels and inspect them

In [38]:
import sys
import importlib
import pathlib
import shutil
import sqlite3
import polars as pl

# Project root — works regardless of where the notebook is opened from
ROOT = pathlib.Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import corruption
importlib.reload(corruption)

DB_CLEAN = str(ROOT / "data" / "order-management.sqlite")
DB = str(ROOT / "data" / "order-management-dirty.sqlite")

shutil.copy2(DB_CLEAN, DB)
with sqlite3.connect(DB) as conn:
    affected = corruption.inject_missing_attribute(conn, "object_Products", "weight", count=5)
    conn.commit()

print(f"Dirty database: {DB}")
print(f"Injected missing weight for {len(affected)} products: {affected}")

Dirty database: /Users/ninoka/Development/ocel-healer/data/order-management-dirty.sqlite
Injected missing weight for 5 products: ['Echo', 'Echo Show 8', 'Echo Studio', 'iPad Air', 'Kindle Paperwhite']


In [39]:
conn = sqlite3.connect(DB)
objects = pl.read_database("SELECT * FROM object", conn)
events = pl.read_database("SELECT * FROM event", conn)
conn.close()
print(f"Objects: {len(objects)}, Events: {len(events)}")
objects.group_by("ocel_type").len().sort("len", descending=True)

Objects: 10840, Events: 21008


ocel_type,len
str,u32
"""items""",7659
"""orders""",2000
"""packages""",1128
"""products""",20
"""employees""",18
"""customers""",15


In [40]:
conn = sqlite3.connect(DB)
objects = pl.read_database("SELECT * FROM object", conn)
events = pl.read_database("SELECT * FROM event", conn)
conn.close()
print(objects)
events

shape: (10_840, 2)
┌───────────────────────────┬───────────┐
│ ocel_id                   ┆ ocel_type │
│ ---                       ┆ ---       │
│ str                       ┆ str       │
╞═══════════════════════════╪═══════════╡
│ Echo                      ┆ products  │
│ Echo Show 8               ┆ products  │
│ Danube Pharmaceuticals BV ┆ customers │
│ Wil van der Aalst         ┆ employees │
│ Christine von Dobbert     ┆ employees │
│ …                         ┆ …         │
│ p-661124                  ┆ packages  │
│ p-661125                  ┆ packages  │
│ p-661126                  ┆ packages  │
│ p-661127                  ┆ packages  │
│ p-661128                  ┆ packages  │
└───────────────────────────┴───────────┘


ocel_id,ocel_type
str,str
"""place_o-990001""","""place order"""
"""pick_i-880003""","""pick item"""
"""place_o-990002""","""place order"""
"""place_o-990003""","""place order"""
"""pick_i-880001""","""pick item"""
…,…
"""deliver_p-661127""","""package delivered"""
"""create_p-661128""","""create package"""
"""send_p-661128""","""send package"""


# Issues detection

## N6 - Incorrect Object
(a) complete duplicate including IDs

(b) same content but different IDs

In [41]:
conn = sqlite3.connect(DB)
objects = pl.read_database("SELECT * FROM object", conn)

# (a) totally duplicated: same ocel_id and ocel_type
dupes_a = objects.filter(objects.select(pl.all().is_duplicated()).to_series())
print("(a) Totally duplicated objects:")
print(dupes_a if not dupes_a.is_empty() else "  None found")

# (b) same content but different IDs: check type-specific tables directly
print("\n(b) Same attributes, different IDs:")
for t in objects["ocel_type"].unique().to_list():
    try:
        attrs = pl.read_database(f'SELECT * FROM "object_{t}"', conn)
        attr_cols = [c for c in attrs.columns if c != "ocel_id"]
        dupes_b = attrs.filter(attrs.select(pl.struct(attr_cols).is_duplicated()).to_series())
        if not dupes_b.is_empty():
            print(f"  Type '{t}':")
            print(dupes_b)
        else:
            print(f"  Type '{t}': None found")
    except Exception:
        print(f"  Type '{t}': no attribute table")

conn.close()

(a) Totally duplicated objects:
  None found

(b) Same attributes, different IDs:
  Type 'orders': None found
  Type 'items':
shape: (1_313, 5)
┌──────────┬─────────────────────┬────────────────────┬────────┬─────────┐
│ ocel_id  ┆ ocel_time           ┆ ocel_changed_field ┆ weight ┆ price   │
│ ---      ┆ ---                 ┆ ---                ┆ ---    ┆ ---     │
│ str      ┆ str                 ┆ null               ┆ f64    ┆ f64     │
╞══════════╪═════════════════════╪════════════════════╪════════╪═════════╡
│ i-880001 ┆ 2023-04-03 12:08:18 ┆ null               ┆ 0.78   ┆ 99.99   │
│ i-880002 ┆ 2023-04-03 12:08:18 ┆ null               ┆ 0.78   ┆ 99.99   │
│ i-880009 ┆ 2023-04-03 23:31:23 ┆ null               ┆ 0.483  ┆ 1099.0  │
│ i-880012 ┆ 2023-04-03 23:31:23 ┆ null               ┆ 0.483  ┆ 1099.0  │
│ i-880015 ┆ 2023-04-04 12:30:50 ┆ null               ┆ 0.98   ┆ 129.99  │
│ …        ┆ …                   ┆ …                  ┆ …      ┆ …       │
│ i-887628 ┆ 2024-03-21 15:37:2

### Discussion point:

Other type of duplicates: 
- duplicate meaning but different IDs and some (not important) attributes

False positive:
- same attributes not always mean the duplicated (ex. name, address could be not a duplicate). Needed at least data knowledge for attribute set, that we consider as a duplicate.

## N2 - Missing Object Type

An object has a missing or unknown type (`ocel_type` is NULL or empty)

In [42]:
conn = sqlite3.connect(DB)
objects = pl.read_database("SELECT * FROM object", conn)
conn.close()

missing_type = objects.filter(
    pl.col("ocel_type").is_null() | (pl.col("ocel_type").str.strip_chars() == "")
)
print("(N2) Objects with missing type:")
print(missing_type if not missing_type.is_empty() else "  None found")

(N2) Objects with missing type:
  None found


## N10 - Incorrect E2O Relation

An E2O relation is erroneously logged: 
- (a) relation with existing object → data knowledge needed
- (b) relation with non-existing object: Detected by finding E2O entries whose `ocel_event_id` or `ocel_object_id` has no matching record in the `event` or `object` table.

In [43]:
conn = sqlite3.connect(DB)
e2o = pl.read_database("SELECT * FROM event_object", conn)
events = pl.read_database("SELECT ocel_id FROM event", conn)
objects = pl.read_database("SELECT ocel_id FROM object", conn).unique()
conn.close()

invalid_event = e2o.join(events, left_on="ocel_event_id", right_on="ocel_id", how="anti")
invalid_object = e2o.join(objects, left_on="ocel_object_id", right_on="ocel_id", how="anti")

print("(N10) E2O relations with non-existent event:")
print(invalid_event if not invalid_event.is_empty() else "  None found")

print("\n(N10) E2O relations with non-existent object:")
print(invalid_object if not invalid_object.is_empty() else "  None found")

(N10) E2O relations with non-existent event:
  None found

(N10) E2O relations with non-existent object:
  None found


## Verification on ocel2-p2p-dirty.sqlite

In [44]:
from src.detection.error_detection import (
    detect_missing_attribute_value,
    detect_wrong_attribute_datatype,
)

missing = detect_missing_attribute_value(DB)
print(f"Missing attribute values: {len(missing)} rows")
print(missing.group_by(["object_type", "attribute"]).len().sort("len", descending=True))

wrong = detect_wrong_attribute_datatype(DB)
print(f"\nWrong attribute datatypes: {len(wrong)} rows")
print(wrong.group_by(["object_type", "attribute", "expected_type"]).len().sort("len", descending=True))

Missing attribute values: 5 rows
shape: (1, 3)
┌─────────────┬───────────┬─────┐
│ object_type ┆ attribute ┆ len │
│ ---         ┆ ---       ┆ --- │
│ str         ┆ str       ┆ u32 │
╞═════════════╪═══════════╪═════╡
│ products    ┆ weight    ┆ 5   │
└─────────────┴───────────┴─────┘

Wrong attribute datatypes: 0 rows
shape: (0, 4)
┌─────────────┬───────────┬───────────────┬─────┐
│ object_type ┆ attribute ┆ expected_type ┆ len │
│ ---         ┆ ---       ┆ ---           ┆ --- │
│ str         ┆ str       ┆ str           ┆ u32 │
╞═════════════╪═══════════╪═══════════════╪═════╡
└─────────────┴───────────┴───────────────┴─────┘
